# 📝 JSON Flattening Demo
This notebook loads 3 nested JSON files and flattens them using `pandas.json_normalize`.

## 🔼 Upload Your JSON Files

In [2]:
from google.colab import files
uploaded = files.upload()

import json
import pandas as pd

# Load all uploaded JSONs into a list
json_data = []
for fname in uploaded.keys():
    with open(fname, 'r') as f:
        json_data.append(json.load(f))


Saving sample2.json to sample2 (1).json
Saving sample3.json to sample3 (1).json
Saving sample1.json to sample1 (1).json


## 🚗 Flatten `automobile` list

In [3]:
auto_data = [rec for rec in json_data if 'automobile' in rec]

df_auto = pd.json_normalize(
    auto_data,
    record_path='automobile',
    meta=['id', 'name'],
    record_prefix='auto_',
    errors='ignore'
)

df_auto.to_csv('flattened_automobiles.csv', index=False)
df_auto.head()


,auto_make,auto_fuelType,id,name
0,Hyundai,Gas,102,Bob
1,Toyota,Gas,101,Alice
2,Tesla,electric,101,Alice


## 🏠 Flatten `address.locations` list

In [4]:
df_locations = pd.json_normalize(
    json_data,
    record_path=['address', 'locations'],
    meta=[
        'id',
        'name',
        ['PASS_PRT', 'N_UMBER'],
        ['PASS_PRT', 'C_OUNTRY'],
        ['PASS_PRT', 'E_XPIRY'],
        ['address', 'city'],
        ['address', 'zip']
    ],
    record_prefix='loc_',
    errors='ignore'
)

df_locations.to_csv('flattened_locations.csv', index=False)
df_locations.head()


,loc_type,loc_zip,loc_metadata.isPrimary,loc_metadata.contacts,id,name,PASS_PRT.N_UMBER,PASS_PRT.C_OUNTRY,PASS_PRT.E_XPIRY,address.city,address.zip
0,office,4321,False,"[{'phone': '321-654-0987', 'email': 'bob.offic...",102,Bob,NaN,NaN,NaN,Funderland,54321
1,remote,5678,False,[],102,Bob,NaN,NaN,NaN,Funderland,54321
2,home,1234,True,"[{'phone': '123-456-7890', 'email': 'alice.hom...",101,Alice,NaN,NaN,NaN,Wonderland,12345


## 📞 Flatten `contacts` inside `locations.metadata.contacts`

In [5]:
df_contacts = pd.json_normalize(
    json_data,
    record_path=['address', 'locations', 'metadata', 'contacts'],
    meta=[
        'id',
        'name',
        ['PASS_PRT', 'N_UMBER'],
        ['PASS_PRT', 'C_OUNTRY'],
        ['PASS_PRT', 'E_XPIRY'],
        ['address', 'city'],
        ['address', 'zip']
    ],
    record_prefix='contact_',
    errors='ignore'
)

df_contacts.to_csv('flattened_contacts.csv', index=False)
df_contacts.head()


,contact_phone,contact_email,contact_social,id,name,PASS_PRT.N_UMBER,PASS_PRT.C_OUNTRY,PASS_PRT.E_XPIRY,address.city,address.zip
0,321-654-0987,bob.office@example.com,[],102,Bob,NaN,NaN,NaN,Funderland,54321
1,111-222-3333,None,"[{'platform': 'linkedin', 'handle': 'bob-link'}]",102,Bob,NaN,NaN,NaN,Funderland,54321
2,123-456-7890,alice.home@example.com,"[{'platform': 'facebook', 'handle': 'alice.won...",101,Alice,NaN,NaN,NaN,Wonderland,12345


### ✅ Done! You now have 3 flattened CSVs:
- `flattened_automobiles.csv`
- `flattened_locations.csv`
- `flattened_contacts.csv`

In [7]:
# import json
# import pandas as pd

# # Step 1: Load the 3 JSONs into a list
# with open("sample1.json") as f1, \
#      open("sample2.json") as f2, \
#      open("sample3.json") as f3:
#     json_data = [json.load(f1), json.load(f2), json.load(f3)]

# # Step 2: Flatten only one level using json_normalize
# df = pd.json_normalize(json_data, sep='.')
# print(df.shape)
# df.head()

(3, 9)


,id,name,automobile,PASS_PRT.N_UMBER,PASS_PRT.C_OUNTRY,PASS_PRT.E_XPIRY,address.city,address.zip,address.locations
0,101,Alice,"[{'make': 'Toyota', 'fuelType': 'Gas'}, {'make...",A12345678,wonderland,2030-12-31,Wonderland,12345,"[{'type': 'home', 'zip': '1234', 'metadata': {..."
1,102,Bob,"[{'make': 'Hyundai', 'fuelType': 'Gas'}]",B12345678,Jumpingland,2030-01-31,Funderland,54321,"[{'type': 'office', 'zip': '4321', 'metadata':..."
2,103,Charlie,NaN,NaN,NaN,NaN,Dreamland,99999,[]


In [9]:
import json
import pandas as pd

# Load the JSON files
with open("sample1.json") as f1, \
     open("sample2.json") as f2, \
     open("sample3.json") as f3:
    json_data = [json.load(f1), json.load(f2), json.load(f3)]

# Step 1: Flatten top-level and dicts
df = pd.json_normalize(json_data, sep='.')

# Step 2: Detect max number of automobiles
max_autos = max(len(p.get('automobile', [])) for p in json_data)

# Step 3: Flatten list-of-dict automobile into columns
for i in range(max_autos):
    df[f'automobile.{i}.make'] = df['automobile'].apply(
        lambda x: x[i].get('make') if isinstance(x, list) and len(x) > i else None
    )
    df[f'automobile.{i}.fuelType'] = df['automobile'].apply(
        lambda x: x[i].get('fuelType') if isinstance(x, list) and len(x) > i else None
    )

# Drop original list column
df.drop(columns=['automobile'], inplace=True)

# Final dataframe
df.head()

,id,name,PASS_PRT.N_UMBER,PASS_PRT.C_OUNTRY,PASS_PRT.E_XPIRY,address.city,address.zip,address.locations,automobile.0.make,automobile.0.fuelType,automobile.1.make,automobile.1.fuelType
0,101,Alice,A12345678,wonderland,2030-12-31,Wonderland,12345,"[{'type': 'home', 'zip': '1234', 'metadata': {...",Toyota,Gas,Tesla,electric
1,102,Bob,B12345678,Jumpingland,2030-01-31,Funderland,54321,"[{'type': 'office', 'zip': '4321', 'metadata':...",Hyundai,Gas,None,None
2,103,Charlie,NaN,NaN,NaN,Dreamland,99999,[],None,None,None,None
